# Purged cross-validation: why plain k-fold flatters a forecast

**Plain idea.** A label built from the next 20 days of returns overlaps the
labels of the 19 neighbouring days. In plain k-fold, test days sit next to
training days whose labels share returns with them, so the model is scored
on data it has partly seen. Purging drops training days whose label window
overlaps the test fold; an embargo drops a few more after it (Lopez de
Prado 2018, *Advances in Financial Machine Learning*, ch. 7).

**What this notebook shows.** On data with **no** predictability, shuffled
k-fold (the default in most ML tutorials) finds some; purged, blocked
k-fold does not. A k-nearest-neighbour regression in numpy only.

In [ ]:
import _paths  # noqa: F401  (puts research/ modules on sys.path)
import numpy as np

In [ ]:
rng = np.random.default_rng(11)
n, h, warm = 1500, 20, 60
r = rng.normal(0, 0.01, warm + n + h)          # i.i.d.: nothing to predict
days = np.arange(warm, warm + n)
y = np.array([r[t + 1:t + 1 + h].sum() for t in days])  # overlapping forward label
# features: trailing 5-, 20- and 60-day returns, known at day t
X = np.array([[r[t - k + 1:t + 1].sum() for k in (5, 20, 60)] for t in days])


def knn_r2(tr, te, k=25):
    # k-nearest-neighbour regression: neighbours in feature space are
    # mostly neighbouring days, whose labels overlap the test label
    d = ((X[te][:, None, :] - X[tr][None, :, :]) ** 2).sum(axis=2)
    nearest = np.argsort(d, axis=1, kind="stable")[:, :k]
    pred = y[tr][nearest].mean(axis=1)
    return 1 - ((y[te] - pred) ** 2).sum() / ((y[te] - y[te].mean()) ** 2).sum()

## Task 1: purged folds

Write `purged_train(test, h, embargo)`: all indices not in `test`, and not
within `h` days before the test fold's first day or `h + embargo` days
after its last day.

In [ ]:
def purged_train(test, h, embargo):
    lo, hi = test.min() - h, test.max() + h + embargo
    idx = np.arange(n)
    return idx[(idx < lo) | (idx > hi)]

In [ ]:
shuffled = np.array_split(np.random.default_rng(3).permutation(n), 5)
plain = [knn_r2(np.setdiff1d(np.arange(n), f), np.sort(f)) for f in shuffled]
folds = np.array_split(np.arange(n), 5)
purged = [knn_r2(purged_train(f, h, 10), f) for f in folds]
print(f"shuffled k-fold mean R2 {np.mean(plain):.3f}; purged blocked {np.mean(purged):.3f}")

In [ ]:
f = folds[2]
tr = purged_train(f, h, 10)
assert not np.intersect1d(tr, f).size
assert tr[tr < f.min()].max() == f.min() - h - 1
assert tr[tr > f.max()].min() == f.max() + h + 10 + 1
assert np.mean(plain) > 0.05   # leakage looks like skill
assert np.mean(purged) < 0.0   # purged: no skill, as planted

**What to take away.** The returns are i.i.d., so no forecast can work, yet
the shuffled score is above zero (about 0.12 here). That is leakage, not
skill. Purged, blocked folds score below zero (about -0.12): a model fitted
to noise does worse than the test fold's own mean, which is the honest
answer.